In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, BaseCrossValidator
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

In [2]:

#### TODO move this to data transformation
panel_data['Churns'] = (~panel_data['CompetesNextYear']).astype(int)
panel_data = panel_data.drop(columns = 'CompetesNextYear')

features = ['TimeSinceLastPBYearEnd',
 'NumberOfMeets',
 'Age',
 'Goodlift',
 'AvgMeetsPerYear',
 'ImprovementGradientBetweenYears',
 'Sex',
 'ImprovementAcceleration',
 'ImprovementGradientTwoMeets',
 'TimeCompetingYearEnd',
 'LastMeetAttemptsMade',
 'FederationProportion']

#Year needed for timeaware split
panel = panel_data.loc[:,features + ['Churns'] + ['Date']  +['Year']]
panel['Date'] = pd.to_datetime(panel['Date'])
panel = panel.sort_values('Date', ascending= True)
panel = panel.drop(columns = 'Date')

In [3]:

#note the train set inclides the train AND validate from the feature selection notebook
train = panel[panel['Year']<=2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'Churns')
train_y = train['Churns']

test_X = test.drop(columns = 'Churns')
test_y = test['Churns']

Note that early stopping was tried but model performed better without so will tune max_iter instead in GridSearchCV

In [4]:
class YearBasedTimeSeriesSplit(BaseCrossValidator):
    #see https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/model_selection/_split.py
    # inheriting from BaseCrossValidator to ensure compatibility with GridSearchCV
    
    def __init__(self, year_column='Year', exclude_val_years = None):
        self.year_column = year_column
        self.exclude_val_years = exclude_val_years or ()

    def split(self, X, y=None, groups=None):
        #needed to overwrite split method in parent class to make sure splits are time aware
        #otherwise the training rows are just all rows that are not validation rows which would create leakage in hyperparam tuning
        
        years = sorted(X[self.year_column].unique())
        
        for i in range(1, len(years)):
            val_year = years[i]
            if val_year in self.exclude_val_years:
                continue
            train_years = years[:i]
            train_idx = np.where(X[self.year_column].isin(train_years))[0]
            val_idx = np.where(X[self.year_column] == val_year)[0]

            #parent class uses yield in implementation of split() so will use here
            yield train_idx, val_idx
        
    def _iter_test_indices(self, X=None, y=None, groups=None):
        #BaseCrossValidator expects _iter_test_indices or _iter_test_masks
        #but since split had to be overwritten, logic for _iter_test_indices is same as for split().
        #(usually split calls _iter_test_masks which calls _iter_test_indices but here it makes more sense to call split for _iter_test_indices)
        for _, val_idx in self.split(X):
            yield val_idx
            
    def get_n_splits(self, X=None, y=None, groups=None):
        #returns the number of folds the splitter will generate. 
        years = X[self.year_column].unique()
        #checks which years in excluded_years are actually present in the dataset
        excl_length = len([y for y in years if y in self.exclude_val_years])
        return len(years) - 1 - excl_length

In [7]:
year_cv = YearBasedTimeSeriesSplit(year_column='Year')

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_iter': [200, 500, 1000],
    'max_depth': [3, 6, 9],
    'min_samples_leaf': [10, 50],
    'l2_regularization': [0.0, 0.1]
}
grid_search = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42, categorical_features = ['Sex']),
    param_grid,
    scoring='accuracy',
    cv=year_cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(train_X, train_y)

Fitting 8 folds for each of 108 candidates, totalling 864 fits


KeyboardInterrupt: 

In [5]:
results_df = pd.DataFrame(grid_search.cv_results_)
best_max_iter = grid_search.best_params_['max_iter']
best_learning_rate = grid_search.best_params_['learning_rate']
best_max_depth = grid_search.best_params_['max_depth']

NameError: name 'grid_search' is not defined

In [ ]:
view = results_df[results_df['param_max_iter'] == best_max_iter]
view

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# fix max_iter at the best value, plot learning_rate vs max_depth to see where to expand grid search
best_max_iter_rows = results_df[results_df['param_max_iter'] == best_max_iter]
pivot1 = best_max_iter_rows.pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_depth'
)
sns.heatmap(pivot1, annot=True, fmt='.3f', ax=axes[0])
axes[0].set_title(f'Fixed max_iter={best_max_iter}')


# fix learning_rate, plot max_iter vs max_depth
best_learning_rate_rows = results_df[results_df['param_learning_rate'] == best_learning_rate]
pivot2 = best_learning_rate_rows.pivot_table(
    values='mean_test_score',
    index='param_max_iter',
    columns='param_max_depth'
)
sns.heatmap(pivot2, annot=True, fmt='.3f', ax=axes[1])
axes[1].set_title(f'Fixed learning_rate={best_learning_rate}')


# fix max_depth, plot learning_rate vs max_iter
best_max_depth_rows = results_df[results_df['param_max_depth'] == best_max_depth]
pivot3 = best_max_depth_rows.pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_iter'
)
sns.heatmap(pivot3, annot=True, fmt='.3f', ax=axes[2])
axes[2].set_title(f'Fixed max_depth={best_max_depth}')

plt.suptitle('Grid Search Results')
plt.tight_layout()
plt.show()

In [ ]:
# best_index = grid_search.best_index_
# n_splits = grid_search.n_splits_

# for i in range(n_splits):
#     score = grid_search.cv_results_[f'split{i}_test_score'][best_index]
#     print(f"Fold {i+1}: {score:.4f}")

# print(f"Mean: {grid_search.cv_results_['mean_test_score'][best_index]:.4f}")
# print(f"Std:  {grid_search.cv_results_['std_test_score'][best_index]:.4f}")

In [ ]:
fine_param_grid = {
    'learning_rate': [0.005, 0.01, 0.03],
    'max_iter': [350, 500, 650,850],
    'max_depth': [2,3, 4, 5]
}

grid_search_fine = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42, categorical_features = ['Sex']),
    fine_param_grid,
    scoring='accuracy',
    cv=year_cv,
    n_jobs=-1,
    verbose=1
)

grid_search_fine.fit(train_X, train_y)

In [ ]:
results_df_fine = pd.DataFrame(grid_search_fine.cv_results_)
best_max_iter = grid_search_fine.best_params_['max_iter']
best_learning_rate = grid_search_fine.best_params_['learning_rate']
best_max_depth = grid_search_fine.best_params_['max_depth']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# fix max_iter at the best value, plot learning_rate vs max_depth to see where to expand grid search
best_max_iter_rows = results_df_fine[results_df_fine['param_max_iter'] == best_max_iter]
pivot1 = best_max_iter_rows.pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_depth'
)
sns.heatmap(pivot1, annot=True, fmt='.3f', ax=axes[0])
axes[0].set_title(f'Fixed max_iter={best_max_iter}')


# fix learning_rate, plot max_iter vs max_depth
best_learning_rate_rows = results_df_fine[results_df_fine['param_learning_rate'] == best_learning_rate]
pivot2 = best_learning_rate_rows.pivot_table(
    values='mean_test_score',
    index='param_max_iter',
    columns='param_max_depth'
)
sns.heatmap(pivot2, annot=True, fmt='.3f', ax=axes[1])
axes[1].set_title(f'Fixed learning_rate={best_learning_rate}')


# fix max_depth, plot learning_rate vs max_iter
best_max_depth_rows = results_df_fine[results_df_fine['param_max_depth'] == best_max_depth]
pivot3 = best_max_depth_rows.pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_iter'
)
sns.heatmap(pivot3, annot=True, fmt='.3f', ax=axes[2])
axes[2].set_title(f'Fixed max_depth={best_max_depth}')

plt.suptitle('Grid Search Results')
plt.tight_layout()
plt.show()

In [ ]:
best_index = grid_search.best_index_
n_splits = grid_search.n_splits_

for i in range(n_splits):
    score = grid_search.cv_results_[f'split{i}_test_score'][best_index]
    print(f"Fold {i+1}: {score:.4f}")

print(f"Mean: {grid_search.cv_results_['mean_test_score'][best_index]:.4f}")
print(f"Std:  {grid_search.cv_results_['std_test_score'][best_index]:.4f}")

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# fix max_iter, plot learning_rate vs max_depth
pivot1 = results_df[results_df['param_max_iter'] == best_max_iter].pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_depth'
)
sns.heatmap(pivot1, annot=True, fmt='.3f', ax=axes[0])
axes[0].set_title(f'Fixed max_iter={best_max_iter}')

# fix learning_rate, plot max_iter vs max_depth
pivot2 = results_df[results_df['param_learning_rate'] == best_learning_rate].pivot_table(
    values='mean_test_score',
    index='param_max_iter',
    columns='param_max_depth'
)
sns.heatmap(pivot2, annot=True, fmt='.3f', ax=axes[1])
axes[1].set_title(f'Fixed learning_rate={best_learning_rate}')

# fix max_depth, plot learning_rate vs max_iter
pivot3 = results_df[results_df['param_max_depth'] == best_max_depth].pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_iter'
)
sns.heatmap(pivot3, annot=True, fmt='.3f', ax=axes[2])
axes[2].set_title(f'Fixed max_depth={best_max_depth}')

plt.suptitle('Grid Search Results - F1', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

fine_best_max_iter = grid_search_f1_2.best_params_['max_iter']
fine_best_learning_rate = grid_search_f1_2.best_params_['learning_rate']
fine_best_max_depth = grid_search_f1_2.best_params_['max_depth']

fine_results_df = pd.DataFrame(grid_search_f1_2.cv_results_)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pivot1 = fine_results_df[fine_results_df['param_max_iter'] == fine_best_max_iter].pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_depth'
)
sns.heatmap(pivot1, annot=True, fmt='.3f', ax=axes[0])
axes[0].set_title(f'Fixed max_iter={fine_best_max_iter}')

pivot2 = fine_results_df[fine_results_df['param_learning_rate'] == fine_best_learning_rate].pivot_table(
    values='mean_test_score',
    index='param_max_iter',
    columns='param_max_depth'
)
sns.heatmap(pivot2, annot=True, fmt='.3f', ax=axes[1])
axes[1].set_title(f'Fixed learning_rate={fine_best_learning_rate}')

pivot3 = fine_results_df[fine_results_df['param_max_depth'] == fine_best_max_depth].pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_iter'
)
sns.heatmap(pivot3, annot=True, fmt='.3f', ax=axes[2])
axes[2].set_title(f'Fixed max_depth={fine_best_max_depth}')

plt.suptitle('Grid Search Results - F1 (Fine)', fontsize=14)
plt.tight_layout()
plt.show()



In [ ]:
param_grid = {
    'learning_rate': [0.125, 0.15, 0.175],
    'max_iter': [800, 900],
    'max_depth': [2,3]
}

grid_search_f1_3 = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=year_cv,
    n_jobs=-1,
    verbose=1
)

grid_search_f1_3.fit(train_X, train_y)

In [ ]:
import matplotlib.pyplot as plt

fine2_best_max_iter = grid_search_f1_3.best_params_['max_iter']
fine2_best_learning_rate = grid_search_f1_3.best_params_['learning_rate']
fine2_best_max_depth = grid_search_f1_3.best_params_['max_depth']

fine2_results_df = pd.DataFrame(grid_search_f1_3.cv_results_)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pivot1 = fine2_results_df[fine2_results_df['param_max_iter'] == fine2_best_max_iter].pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_depth'
)
sns.heatmap(pivot1, annot=True, fmt='.3f', ax=axes[0])
axes[0].set_title(f'Fixed max_iter={fine2_best_max_iter}')

pivot2 = fine2_results_df[fine2_results_df['param_learning_rate'] == fine2_best_learning_rate].pivot_table(
    values='mean_test_score',
    index='param_max_iter',
    columns='param_max_depth'
)
sns.heatmap(pivot2, annot=True, fmt='.3f', ax=axes[1])
axes[1].set_title(f'Fixed learning_rate={fine2_best_learning_rate}')

pivot3 = fine2_results_df[fine2_results_df['param_max_depth'] == fine2_best_max_depth].pivot_table(
    values='mean_test_score',
    index='param_learning_rate',
    columns='param_max_iter'
)
sns.heatmap(pivot3, annot=True, fmt='.3f', ax=axes[2])
axes[2].set_title(f'Fixed max_depth={fine2_best_max_depth}')

plt.suptitle('Grid Search Results - F1 (Fine 2)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
results_df = pd.DataFrame(grid_search_f1_3.cv_results_)

# get scores for each fold for the best parameter combination
best_index = grid_search_f1_3.best_index_

fold_scores = {col: results_df.loc[best_index, col] 
               for col in results_df.columns 
               if col.startswith('split')}

for fold, score in fold_scores.items():
    print(f"{fold}: {score:.4f}")

In [ ]:
best_clf = HistGradientBoostingClassifier(
    **grid_search_f1_3.best_params_,
    random_state=42
)
best_clf.fit(full_train_X, full_train_y)